# 第 08 章　文件操作

> **本章定位**：让数据活过一个程序的运行周期。日志按行追加、日报表按 CSV 导出、历史数据按日期归档。
>
> **知识点范围**：`open()` 与三种模式（`r` / `w` / `a`）、`read()` / `readline()` / `readlines()` / 逐行遍历、`write()` / `writelines()`、`with` 上下文管理、`encoding="utf-8"`、CSV 文本的写入与解析、`close()` 的必要性。
>
> **本章约定**：可用前 7 章全部内容；**不引入** `os`、`csv`、`json` 等模块（模块在第 10 章）。CSV 采用 `split(",")` / `",".join()` 手工处理。

---

## 本章学习路径

| 阶段 | 内容 | 编号 | 作用 |
|---|---|---|---|
| 第一部分 | 知识讲解 | 知识点 8.1 — 8.5 | 打开、读写、关闭、CSV + 随堂示例 |
| 第二部分 | 代码示范 | 案例 8-1、案例 8-2 | 日志统计 / CSV 报表 |
| 第三部分 | 布置练习 | 练习 8-1、练习 8-2 | 分时汇总 / 告警追加 |
| 第四部分 | 习题答案 | 答案 8-1、答案 8-2 | 对照解析与评分要点 |

**知识点与代码对照表**：

| 知识点 | 名称 | 讲解位置 | 代码示范 | 对应练习 |
|---|---|---|---|---|
| 8.1 | `open()` 与三种模式 | 8.2 | 案例 8-1、案例 8-2 | 练习 8-1、练习 8-2 |
| 8.2 | 三种读取方式与逐行遍历 | 8.3 | 案例 8-1 | 练习 8-1 |
| 8.3 | 写入：`write()` 与换行 | 8.4 | 案例 8-1、案例 8-2 | 练习 8-1、练习 8-2 |
| 8.4 | `with` 与 `encoding` | 8.5 | 两个案例 | 两个练习 |
| 8.5 | CSV 手工写入与解析 | 8.6 | 案例 8-2 | 练习 8-1 |

## 教学提示（教师用）

- **建议课时**：8 学时（4 次课），对应课程标准模块 8。
- **讲解主线**：文件操作永远是三步：开 → 读 / 写 → 关。`with` 与 `encoding="utf-8"` 是纪律而非可选项；先讲读，再讲写，最后讲 CSV 的手工解析（不引入第三方库）。
- **学生常见卡点**：用 `w` 打开已有文件会**清空**原内容；写入时忘记 `"
"` 导致多行粘连；读回后忘记 `strip()`，`int("62
")` 抛 `ValueError`；相对路径依赖当前工作目录，换目录即找不到文件。
> - **课堂活动建议**：故意把告警文件用 `w` 打开一次，观察历史数据消失，再改回 `a` 追加，用"数据丢了"的直观冲击建立模式意识。

---

# 第一部分　知识讲解

## 8.1 场景导入：三件事，两种事故

文件操作的核心只有三件事：**打开、读写、关闭**。但两类事故极常见：

1. **模式选错**：本想追加一条日志，用了 `w`，把历史数据全部清空；
2. **忘记关闭**：数据还留在缓冲区，程序结束或断电后文件内容缺失。

`with` 语句同时解决第二个问题：无论中途是否异常，文件都会被正确关闭。

## 8.2 知识点 8.1　`open()` 与三种模式

### 8.2.1 核心概念

`open(文件名, 模式, encoding="utf-8")` 返回一个文件对象。模式决定"能做什么"：

| 模式 | 含义 | 文件不存在时 | 是否清空原内容 | 典型用途 |
|---|---|---|---|---|
| `"r"` | 只读 | 抛 `FileNotFoundError` | —— | 读取日志、报表 |
| `"w"` | 只写 | 自动创建 | **是，清空重写** | 生成新报表 |
| `"a"` | 追加写 | 自动创建 | 否，写到末尾 | 写日志、追加告警 |
| `"r+"` | 读写 | 抛错 | 否 | 需要原地修改（初学不建议） |

**原理补充（概念深化）**：`open(path, "w")` 在真正写入前，内核先把该文件截断（truncate）为 0 字节，这是「只写模式」的定义使然，而非 Python 的额外动作；因此「用 `w` 追加日志」必然丢历史，日志只能用 `"a"`：追加模式把文件指针固定在末尾，且每次 `write` 前都不会截断。`encoding="utf-8"` 决定的是「字符串与字节」之间的翻译表：写时把 Unicode 字符编码成字节序列，读时按同一张表解码；两侧不一致就 `UnicodeDecodeError` 或乱码，Windows 默认 GBK 尤其容易踩坑。`write` 还常先落在用户态缓冲区，`close()` 或 `with` 退出时才 flush 到磁盘，忘关可能丢尾部数据。本节不碰磁盘，用「一个字符串变量当文件内容」复现 `w` 覆盖、`a` 追加、`r` 只读三种模式对同一份内容的结果差异。

### 8.2.2 语法要点

| 要素 | 规范 | 说明 |
|---|---|---|
| 文件名 | 相对路径（同目录）或绝对路径 | 相对路径以"当前工作目录"为基准 |
| 模式 | 字符串，如 `"r"` | 省略时默认 `"r"` |
| 编码 | `encoding="utf-8"` | 读写中文**必须**显式指定 |
| 返回值 | 文件对象 | 后续调用 `.read()` / `.write()` |

### 8.2.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 用 `w` 写日志 | 历史记录全部丢失 | 日志一律用 `a` |
| 忘记 `encoding="utf-8"` | Windows 下中文乱码或 `UnicodeDecodeError` | 每次 `open` 都写 |
| 路径用了单反斜杠 | `\t` 被当制表符 | 用 `/` 或 `\\`（第 1 章知识点 1.3） |
| 读不存在的文件 | `FileNotFoundError` | 先确认文件存在，或用第 9 章 `try` |
| Jupyter 中找不到文件 | 工作目录不是章节目录 | 用 `os.getcwd()` 确认（第 10 章） |

### 8.2.4 随堂示例：w / a / r 三种模式的实际差别

**核心要点**：用同一文件连续演示三种模式：`w` 清空后写入、`a` 追加到末尾、`r` 读取；关键在于 `w` 会覆盖原内容，`a` 不会。

In [ ]:
with open("demo8_mode.txt", "w", encoding="utf-8") as f:   # w：新建或清空
    f.write("820\n")

with open("demo8_mode.txt", "a", encoding="utf-8") as f:   # a：末尾追加
    f.write("1100\n")

with open("demo8_mode.txt", "r", encoding="utf-8") as f:   # r：读取
    print("文件内容：")
    print(f.read(), end="")

**预期输出**：

```text
文件内容：
820
1100
```

**变式代码（迁移练习）**：相对随堂示例用真实文件依次 `w`/`a`/`r`，本变式用一个字符串变量模拟「同一份文件内容」，复现 `w` 覆盖、`a` 追加、`r` 只读切分三种模式的结果差异。

In [ ]:
# 用字符串 content 模拟磁盘上的文件内容
content = "06:00,820\n"                 # 假定原有 1 行
content = "08:00,1100\n"                # w：清空重写，原内容被覆盖
content = content + "10:00,760\n"       # a：末尾追加，历史不丢
lines = content.strip().split("\n")     # r：只读地按行切分
print("最终内容：")
print(content, end="")
print("只读统计：", len(lines), "行，首行", lines[0])

**预期输出**：

```text
最终内容：
08:00,1100
10:00,760
只读统计： 2 行，首行 08:00,1100
```

### 8.2.5 实操步骤（上机跟随）

1. `content` 起始是 `06:00,820` 加一个换行符，随后被整体重新赋值为 `08:00,1100`，模拟 `w` 清空重写——原来的 06:00 一行消失
2. 再执行追加 `content = content + ...`，模拟 `a` 末尾写入，08:00 与 10:00 两行同时保留
3. 用 `strip()` 去掉首尾空白后按换行符切分得到 2 行列表，模拟 `r` 只读读取，不改动内容本身
4. 打印最终内容为 08:00、10:00 两行，核对首行 `08:00,1100`，验证 w 已覆盖而 a 没有覆盖
5. 把追加那步误写成覆盖赋值，最终只剩 1 行，直观看到「日志丢失」是怎么发生的

### 8.2.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| 用 `w` 追加日志，历史全没了 | `w` 模式在写入前把文件截断为 0 字节 | 日志一律用 `a` 模式打开 |
| 读不存在的文件报 `FileNotFoundError` | `r` 模式要求文件已存在 | 确认路径与文件存在，或第 9 章用 `try` 兜底 |
| Windows 下中文乱码 / `UnicodeDecodeError` | 读写两侧编码不一致或漏写 encoding | 每次 `open` 都显式写 `encoding="utf-8"` |
| 路径中的反斜杠被当成转义符（如制表符） | 单反斜杠触发转义 | 路径改用正斜杠 `/`，或写成双反斜杠 |
| Jupyter 里找不到文件 | 相对路径以当前工作目录为基准，未必是章节目录 | 先确认工作目录，或改用明确的相对路径 |

## 8.3 知识点 8.2　三种读取方式

### 8.3.1 核心概念

| 方法 | 返回 | 适用 | 内存 |
|---|---|---|---|
| `f.read()` | 整个文件为一个字符串 | 小文件、需要整体处理 | 全量载入 |
| `f.readline()` | 一行（含 `\n`） | 逐行控制、需要中途停止 | 一行 |
| `f.readlines()` | 所有行组成的**列表** | 需要按下标访问行、统计行数 | 全量载入 |
| `for line in f:` | 每次一行 | **大文件首选** | 一行 |

**原理补充（概念深化）**：整段读、单行读、按行读共享同一个文件指针——它是一个字节偏移量：整段读把它推到末尾，单行读每次推进到下一个换行符之后，按行读等价于反复单行读直到末尾再收集成列表。所以「读完再读为空」不是缓存失效，而是指针已停在 EOF，重读必须重新 `open` 或 `seek(0)`。`for line in f` 为何省内存：文件对象自身是迭代器，逐行只把当前一行读进内存，而整段读 / 按行读会把全量内容物化成字符串或列表，大文件下内存暴涨。每行末尾保留的换行符也必须先 `strip()` 再用：带换行的字符串虽能被 `int()` 容忍，但字符串相等判断会失败。本节用一段字符串配合整段引用、按行切分与下标，在纯计算里复现三种读法的返回形态与指针单向前进。

### 8.3.2 语法要点

- 逐行遍历得到的每行**末尾带 `\n`**，必须先 `strip()` 再用。
- 读完一遍后文件指针在末尾，再读得到空串；需要重读必须重新 `open` 或 `seek(0)`。
- 判断空行要在 `strip()` **之后**：`if line == "":` 才能识别只含换行符的行。

### 8.3.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 忘记 `strip()` | 字段末位带 `\n`，比较失败 | 每行先 `strip()` |
| 用 `read()` 处理大文件 | 内存暴涨 | 改用 `for line in f` |
| 遍历两次 | 第二次为空 | 重新打开或用 `readlines()` 缓存 |
| 空行未跳过 | `IndexError` 或统计偏多 | `if line == "": continue` |

### 8.3.4 随堂示例：read / readline / readlines 三种读法

**核心要点**：验证三种读取方式的返回形态：`read()` 得到整段字符串、`readline()` 得到一行、`readlines()` 得到按行切分的列表；文件指针只前进不回退。

In [ ]:
with open("demo8_read.txt", "w", encoding="utf-8") as f:
    f.write("06:00,820\n08:00,1100\n10:00,760\n")

with open("demo8_read.txt", encoding="utf-8") as f:
    print("read() 全文：")
    print(f.read(), end="")

with open("demo8_read.txt", encoding="utf-8") as f:
    print("readline() 首行：", f.readline().strip())
    print("readline() 次行：", f.readline().strip())

with open("demo8_read.txt", encoding="utf-8") as f:
    lines = f.readlines()
print("readlines() 共", len(lines), "行；末行：", lines[-1].strip())

**预期输出**：

```text
read() 全文：
06:00,820
08:00,1100
10:00,760
readline() 首行： 06:00,820
readline() 次行： 08:00,1100
readlines() 共 3 行；末行： 10:00,760
```

**变式代码（迁移练习）**：相对随堂示例对真实文件依次调用三种读法，本变式在一段字符串上用整段引用、按行切分与下标模拟整段读 / 单行读 / 按行读的返回形态，并故意留一行首尾空格验证 `strip()` 的必要性。

In [ ]:
# 用一段字符串模拟文件内容，第二行故意带首尾空格
text = "06:00,820\n 08:00,1100 \n10:00,760\n"
whole = text                            # 整段读：一个字符串
rows = text.strip().split("\n")         # 按行读：切成分行列表
print("整段读全文：")
print(whole, end="")
print("按行读共", len(rows), "行")
first, second = rows[0], rows[1]        # 逐行读：依次取前两行
print("首行读：", first)
print("次行去空格：", "[" + second + "]", "→", "[" + second.strip() + "]")

**预期输出**：

```text
整段读全文：
06:00,820
 08:00,1100 
10:00,760
按行读共 3 行
首行读： 06:00,820
次行去空格： [ 08:00,1100 ] → [08:00,1100]
```

### 8.3.5 实操步骤（上机跟随）

1. `whole = text` 得到整段字符串，模拟整段读一次载入全文（三行都在同一个字符串里）
2. `rows` 经 `strip()` 后按换行符切分得到 3 个元素，模拟按行读返回的分行列表
3. `first, second = rows[0], rows[1]` 模拟连续两次单行读，指针每次只前进一行
4. 打印 `[ 08:00,1100 ] → [08:00,1100]`，验证行首尾空格必须 `strip()` 后才能正确比较
5. 整段读会全量物化，逐行遍历只占一行内存，因此处理大文件首选 `for line in f`

### 8.3.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| `int(...)` 或字符串比较出错 | 读到的行尾带换行符或首尾空格 | 每行先 `strip()` 再解析或比较 |
| 第二次遍历文件得到空 | 文件指针已停在 EOF，不会自动回退 | 重新 `open` 或 `seek(0)`，或先缓存成列表 |
| 处理大文件时内存暴涨 | 整段读 / 按行读会把全量载入内存 | 改用 `for line in f` 逐行处理 |
| 空行让 `parts[2]` 越界报 `IndexError` | 空行 `strip()` 后为空串，仍被当成记录解析 | 先 `strip()`，再用 `if line == "": continue` 跳过 |

## 8.4 知识点 8.3　写入

### 8.4.1 核心概念

- `f.write(文本)`：写入一个字符串，**不自动换行**，需要自己加 `"\n"`；
- `f.writelines(列表)`：把列表中每个字符串依次写入，同样**不自动加换行**；
- 写入内容必须是**字符串**，数字要先 `str()` 或 `format`。

**原理补充（概念深化）**：文件是字节流，本身没有「行」的概念，换行符只是一个普通字符；`write(s)` 只负责把 `s` 编码后的字节追加到当前指针位置，不会自作主张插入分隔符，因此「每行自己补换行符」是调用方的责任，`writelines(list)` 也只是对列表逐个元素调用 `write`，同样不补。写入内容必须是字符串：文本模式下底层只接受 `str`，直接写数字会抛 `TypeError`，数字须先 `str()` 或格式化；二进制模式另说。缓冲也值得留意：写入常先进入用户态缓冲区，`with` 退出或 `close()` 时 flush 才真正落盘，程序异常退出可能丢尾部内容，这正是 `with` 是纪律而非可选项的原因。本节不落磁盘，用字符串拼接模拟「写」：逗号连接构造一行、显式补换行符，再切分回列表模拟读回校验行数。

### 8.4.2 语法要点

| 场景 | 写法 |
|---|---|
| 写一行 | `f.write(line + "\n")` |
| 写多行 | `for line in rows: f.write(line + "\n")` |
| 追加一行到末尾 | 以 `"a"` 模式打开后 `f.write(...)` |
| 写数字 | `f.write(str(total) + "\n")` |

### 8.4.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 忘记 `\n` | 所有记录挤在一行 | 每条加 `"\n"` |
| 直接写数字 | `TypeError: write() argument must be str` | `str()` 转换 |
| 追加却用了 `w` | 原文件被清空 | 改 `"a"` |
| 缓冲区未刷新 | 程序异常退出后内容缺失 | 用 `with`，它负责关闭与刷新 |

### 8.4.4 随堂示例：写入必须自己补换行

**核心要点**：验证 `write()` 不会自动换行：不写 `\n` 时多行会粘连成一行；写完后读回校验行数，是最直接的确认手段。

In [ ]:
rows = ["06:00,820", "08:00,1100", "10:00,760"]

with open("demo8_write.txt", "w", encoding="utf-8") as f:
    for r in rows:
        f.write(r + "\n")        # 每行自己补换行

with open("demo8_write.txt", encoding="utf-8") as f:
    lines = f.readlines()
print("行数：", len(lines))
print("首行：", lines[0].strip())
print("末行：", lines[-1].strip())

**预期输出**：

```text
行数： 3
首行： 06:00,820
末行： 10:00,760
```

**变式代码（迁移练习）**：相对随堂示例真实写文件再读回，本变式用纯字符串拼接模拟 `write()`，对比「每行补换行符得 3 行」与「漏补换行符三行粘连」，并验证数字必须先 `str()`。

In [ ]:
# 用字符串拼接模拟写入：数字先 str、每行自己补换行符
rows = [("06:00", 820), ("08:00", 1100), ("10:00", 760)]
content = ""
for t, v in rows:
    content += t + "," + str(v) + "\n"   # 补换行符，数字已转字符串
lines = content.strip().split("\n")
print("写入内容：\n" + content, end="")
print("读回", len(lines), "行，首行", lines[0])
bad = "".join([t + "," + str(v) for t, v in rows])  # 故意不补换行符
print("漏写换行：", bad)

**预期输出**：

```text
写入内容：
06:00,820
08:00,1100
10:00,760
读回 3 行，首行 06:00,820
漏写换行： 06:00,82008:00,110010:00,760
```

### 8.4.5 实操步骤（上机跟随）

1. `content += t + "," + str(v) + ...` 把数字 820 先 `str()` 再拼接，模拟写入只接受字符串
2. 三条记录各补一个换行符，`strip()` 后按换行符切分读回得到 3 行，首行 `06:00,820`
3. `bad` 用 `join` 故意不补换行符，结果是一整串 `06:00,82008:00,110010:00,760`，直观看到多行粘连
4. 把 `str(v)` 去掉直接拼数字，会触发 `TypeError`，印证写入内容必须是字符串
5. 真实场景里 `writelines(list)` 同样不补换行，且 `with` 退出时才刷新缓冲

### 8.4.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| 多行记录挤成一行 | `write()` 不自动加换行符 | 每行末尾自己补一个换行符 |
| `TypeError: write() argument must be str` | 把数字直接传给写入函数 | 先 `str()` 或格式化成字符串 |
| 追加内容却把旧文件清空了 | 追加误用了 `w` 模式 | 日志 / 追加一律用 `a` 模式 |
| 程序异常退出后尾部内容缺失 | 写入留在缓冲区未 flush | 用 `with` 块，退出时自动关闭并刷新 |

## 8.5 知识点 8.4　`with` 与编码

### 8.5.1 核心概念

```text
with open("文件", "r", encoding="utf-8") as f:
    逐行处理
```

`with` 块结束时（正常结束或抛异常），Python 自动调用 `f.close()`。这比手写 `try/finally` 简洁，也是官方推荐写法。

**原理补充（概念深化）**：`with` 背后是上下文管理协议：`open()` 返回的文件对象是上下文管理器，进入块前调用 `__enter__()`，把返回值绑定到 `as f`；退出块时——无论正常结束、提前 `return`，还是中途抛异常——都保证调用 `__exit__()`，其中执行 `close()` 并 flush 缓冲。相比手写 `try/finally`，异常路径也不会漏关。`as f` 只是把 `__enter__` 的返回值做一次名字绑定，块结束后名字虽还在，底层流却已关闭，此时再调用读取会抛 `ValueError: I/O operation on closed file`。编码为什么两侧都要写：`encoding="utf-8"` 同时规定写入时的编码与读取时的解码，只写一侧，中文在一端被编成 UTF-8 字节、另一端按系统默认（如 GBK）解码就乱码。本节不碰磁盘，用一个状态变量模拟「进入未关闭、退出必定关闭」的两段，编码作为概念说明。

### 8.5.2 语法要点

- `as f` 中的 `f` 只在 `with` 块内有效；
- 可以在一个 `with` 中打开多个文件（用逗号分隔），但不建议超过两个；
- `encoding="utf-8"` 是**读写两侧都要写**的参数，漏写一侧就可能乱码。

### 8.5.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 手写 `close()` 但中途异常 | 文件未关闭、数据可能丢失 | 统一用 `with` |
| 在 `with` 块外使用 `f` | `ValueError: I/O operation on closed file` | 处理逻辑写在块内 |
| 只写一半编码 | 写入正常、读取乱码 | 读写都写 `encoding="utf-8"` |
| 用 `w` 打开后立刻读 | `io.UnsupportedOperation: not readable` | 模式与操作要匹配 |

### 8.5.4 随堂示例：with 自动关闭与 encoding 防乱码

**核心要点**：验证 `with` 语句块结束时自动关闭文件（`f.closed` 变为 `True`），以及 `encoding=「utf-8」` 保证中文读写一致。

In [ ]:
with open("demo8_with.txt", "w", encoding="utf-8") as f:
    f.write("路段,流量\n京开,1100\n")
    print("块内是否关闭：", f.closed)

print("块外是否关闭：", f.closed)          # with 结束自动关闭

with open("demo8_with.txt", encoding="utf-8") as f:
    print(f.read(), end="")

**预期输出**：

```text
块内是否关闭： False
块外是否关闭： True
路段,流量
京开,1100
```

**变式代码（迁移练习）**：相对随堂示例用真实 `f.closed` 观察 with 自动关闭，本变式用状态变量与 `enter` 函数模拟上下文管理器的进入 / 退出两段，验证「块内未关闭、块外已关闭」，编码一致性作为概念写在原理补充。

In [ ]:
# 用状态变量模拟 with：进入块打开、退出块自动关闭
state = {"closed": True}
def enter(note):
    state["closed"] = False                # 进入块：文件处于打开状态
    return note
f = enter("京开,1100")                     # 类比 with 把返回值绑定给 f
print("块内 closed：", state["closed"], "| 内容：", f)
state["closed"] = True                     # 退出块：自动关闭并刷新
print("块外 closed：", state["closed"], "| 读写两侧编码：utf-8")

**预期输出**：

```text
块内 closed： False | 内容： 京开,1100
块外 closed： True | 读写两侧编码：utf-8
```

### 8.5.5 实操步骤（上机跟随）

1. `enter()` 把 `state["closed"]` 设为 `False` 并返回内容，模拟 `with` 进入块时上下文管理器打开文件
2. 块内打印 `closed： False`，对应随堂示例里 `f.closed` 为 `False`
3. 把状态改回 `True` 模拟退出时自动 `close()` 与 flush，块外打印 `closed： True`
4. 打印 `读写两侧编码：utf-8`，提醒写端与读端必须用同一张编码表，否则中文乱码
5. 真实 `with` 即便块内抛异常也会执行退出逻辑，这是它比手写 `close()` 安全的原因（见原理补充）

### 8.5.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| `ValueError: I/O operation on closed file` | 在 `with` 块外继续使用 `f`，底层流已关闭 | 所有读写逻辑都写在 `with` 块内 |
| 手写 `close()` 但中途异常，文件没关 | 异常打断正常流程，`close()` 未执行 | 统一用 `with`，退出逻辑在异常路径也保证关闭 |
| 写入正常、读取却乱码 | 只在 `open` 的一侧写了 `encoding="utf-8"` | 读、写两侧都显式指定 `encoding="utf-8"` |
| `io.UnsupportedOperation: not readable` | 用 `w` 打开后立刻调用读取 | 模式与操作匹配：要读就用 `r` 或默认模式 |

## 8.6 知识点 8.5　CSV 手工写入与解析

### 8.6.1 核心概念

CSV（Comma-Separated Values）本质是"逗号分隔的纯文本"，**不是 Excel 文件**。它没有格式、没有公式，只有行与逗号。

- **写**：把字段转成字符串，用 `",".join()` 拼成一行；
- **读**：`split(",")` 拆成字段，数字字段再 `int()` / `float()`；
- **表头**：第一行是字段名，统计时要跳过。

**原理补充（概念深化）**：CSV 没有类型，它只是「字段用逗号分隔、记录用换行分隔」的纯文本，所有值写进去都是字符串，所以逗号连接要求每个元素已是 `str`（数字须 `map(str, ...)`），读回按逗号切分拿到的也全是字符串，必须 `int()` / `float()` 才能参与计算。为什么要跳过表头：表头行是字段名字符串，若直接把第二列喂给 `int` 会把「流量」这样的文字转换而抛 `ValueError`；用 `isdigit()` 判断或标志变量 `first` 跳过。手工解析也有边界：按逗号切分无法处理字段内自带逗号（需要引号转义），这正是本章手工法与第 10 章 `csv` 模块的分工；写侧连接、读侧切分互为逆运算，体现的正是「分隔符加换行」这一本质。本节同样用字符串模拟整份 CSV 文本，不落磁盘。

### 8.6.2 语法要点

| 操作 | 写法 | 说明 |
|---|---|---|
| 拼一行 | `",".join(["06:00", "820", "68"])` | 元素必须都是字符串 |
| 数字转字符串 | `",".join(map(str, row))` | `map(str, 元组)`（第 7 章） |
| 跳过表头 | 首次循环 `continue`，或用标志变量 | 两种写法案例 8-2 均演示 |
| 追加汇总行 | 用 `"a"` 模式再 `write` | 不可用 `w` |

### 8.6.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| `join` 里含数字 | `TypeError: sequence item 1: expected str` | `map(str, ...)` |
| 字段本身含逗号 | 列数错乱 | 真实项目用 `csv` 模块处理引号转义 |
| 表头未跳过 | 统计时 `int("车流量")` 报错 | 跳过第一行 |
| 用 Excel 打开乱码 | 显示异常 | 编码问题，用 UTF-8 或加 BOM |

> **本章与第 10 章的分工**：本章手工处理 CSV，目的是理解"分隔符 + 换行"的本质；第 10 章会引入标准库 `csv`，处理引号、转义等复杂情况。

---

### 8.6.4 随堂示例：手工拼 CSV 并逐行解析

**核心要点**：验证 CSV 的本质：用逗号连接字段、每行一条记录。写出时 `join()` 拼行，读回时 `split(\「,\」)` 拆字段，无需第三方库。

In [ ]:
rows = [("时段", "流量"), ("06:00", 820), ("08:00", 1100)]

with open("demo8_csv.csv", "w", encoding="utf-8") as f:
    for r in rows:
        f.write(",".join(map(str, r)) + "\n")

total = 0
with open("demo8_csv.csv", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split(",")
        print("字段：", parts)
        if parts[1].isdigit():        # 跳过表头
            total += int(parts[1])

print("流量合计：", total)

**预期输出**：

```text
字段： ['时段', '流量']
字段： ['06:00', '820']
字段： ['08:00', '1100']
流量合计： 1920
```

**变式代码（迁移练习）**：相对随堂示例借真实文件演示连接 / 切分，本变式把整份 CSV 折叠成一个字符串变量，用按行切分加逐行按逗号切分复现解析，验证 `map(str, ...)` 拼字段、`isdigit()` 跳表头、`int()` 累加。

In [ ]:
# 用字符串模拟 CSV：join 拼行、split 解析，不落磁盘
rows = [("时段", "流量"), ("06:00", 820), ("08:00", 1100), ("10:00", 760)]
text = "\n".join([",".join(map(str, r)) for r in rows])   # 每行先拼字段
print("模拟 CSV 文本：")
print(text)
total = 0                                  # 累加器在循环外
for line in text.split("\n"):
    parts = line.split(",")
    total += int(parts[1]) if parts[1].isdigit() else 0   # 跳过表头
print("流量合计：", total)

**预期输出**：

```text
模拟 CSV 文本：
时段,流量
06:00,820
08:00,1100
10:00,760
流量合计： 2680
```

### 8.6.5 实操步骤（上机跟随）

1. 用逗号连接字段时，数字 820 必须先经 `map(str, ...)` 转成字符串，得到一行 `06:00,820`
2. 再用换行符把 4 行拼成整段文本，第一行是表头 `时段,流量`
3. 逐行按逗号切分后取第 2 个字段，`isdigit()` 对表头「流量」返回 `False` 故跳过
4. 三条数据行分别累加 820、1100、760，输出 `流量合计： 2680`
5. 若字段本身含逗号（如 `京港澳,北,820`），按逗号切分会多出列，需第 10 章 `csv` 模块处理

### 8.6.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| `TypeError: sequence item 1: expected str` | `join` 的元素里混了数字 | 用 `map(str, r)` 或先 `str()` 再 `join` |
| `int(...)` 抛 `ValueError` | 表头行没跳过，字段名被当成数字 | 用 `isdigit()` 判断或标志变量跳过第一行 |
| 字段含逗号导致列数错乱 | 手工按逗号切分不处理引号转义 | 本章保证字段不含逗号；真实项目用第 10 章 `csv` 模块 |
| 读回的数字相加变成字符串拼接 | 按逗号切分拿到的全是字符串，没 `int()` | 数值字段先 `int()` / `float()` 再运算 |
| 用 Excel 打开中文 CSV 乱码 | Excel 按系统默认编码解码 | 用 UTF-8 保存，必要时加 BOM |

# 第二部分　代码示范

## 案例 8-1　车载日志写入与逐行统计

- **覆盖知识点**：8.1（`w` / `r`）、8.3（`write` 换行）、8.2（逐行遍历、`strip`、`split`）、8.4（`with` + `encoding`）
- **源文件**：`case1_日志读写统计.py`
- **难度**：基础　**建议用时**：30 分钟

### 任务情境

车载终端每 30 秒上传一条记录，格式为 `时间|车牌|车速|车道`。需要先把一个班次的记录写入日志文件，再读回统计总条数、超速条数与平均车速。

### 任务要求

1. 用 `w` 模式写入 6 条样例记录，每条以 `\n` 结尾；
2. 用 `with open(..., encoding="utf-8") as f` 读取，逐行遍历；
3. 每行用 `strip()` + `split("|")` 解析出 4 个字段；
4. 统计总条数、超速条数（车速 > 120）、平均车速；
5. 输出时保留必要的小数位。

### 参考代码

In [ ]:
# ---------- 第 08 章 文件操作 · 案例 8-1：车载日志写入与逐行统计 ----------
# 对应知识点：8.1 w / r 模式、8.3 write() 与换行、8.2 逐行遍历与 strip() / split()、8.4 with + encoding
# 功能说明：写入 6 条样例车载日志，再逐行读回统计记录数、平均车速并打印超速记录

# ---------- 1) 写入样例日志 ----------
records = [
    "2026-10-29 08:15:32|京A12345|62|2",
    "2026-10-29 08:16:02|京B67890|118|1",
    "2026-10-29 08:16:32|津C24680|132|3",
    "2026-10-29 08:17:02|冀D11223|78|2",
    "2026-10-29 08:17:32|京E55667|150|1",
    "2026-10-29 08:18:02|京A13579|95|3",
]

with open("vehicle_log.txt", "w", encoding="utf-8") as f:
    for line in records:
        f.write(line + "\n")          # 每行必须以换行结尾
print("已写入", len(records), "条记录")

# ---------- 2) 读取并统计 ----------
total = 0
over = 0
speed_sum = 0

with open("vehicle_log.txt", "r", encoding="utf-8") as f:
    for line in f:                    # 逐行读取，内存占用最小
        line = line.strip()
        if line == "":                # 跳过空行，防止解析越界
            continue
        parts = line.split("|")
        speed = int(parts[2])         # 字符串转整数
        total += 1
        speed_sum += speed
        if speed > 120:
            over += 1
            print("  超速：", parts[1], speed, "km/h")

print("总记录数：", total, "条")
print("超速记录：", over, "条")
print("平均车速：", round(speed_sum / total, 1), "km/h")

### 运行结果

```text
已写入 6 条记录
  超速： 津C24680 132 km/h
  超速： 京E55667 150 km/h
总记录数： 6 条
超速记录： 2 条
平均车速： 105.8 km/h
```

### 代码解读

| 代码段 | 对应知识点 | 作用 |
|---|---|---|
| `open("vehicle_log.txt", "w", encoding="utf-8")` | 8.1、8.4 | 写模式 + UTF-8；文件不存在会自动创建 |
| `f.write(line + "\n")` | 8.3 | `write` 不自动换行，必须自己加 |
| `with … as f:` | 8.4 | 块结束自动关闭，异常也安全 |
| `for line in f:` | 8.2 | 逐行迭代，内存只占一行 |
| `line.strip()` | 8.2、4.2 | 去掉行尾 `\n`，否则 `int("132\n")` 虽能转但比较易出错 |
| `if line == "": continue` | 8.2 | 跳过空行，防止 `parts[2]` 越界 |
| `line.split("\|")` | 4.4、8.5 | 按分隔符拆字段 |
| `int(parts[2])` | 2.3、8.5 | 文本转数字才能比较与累加 |
| `total / over / speed_sum` | 3.5 | 三个累加器在循环外初始化 |

**改一改**：把第一次 `open` 的模式改成 `"a"`，连续运行两次，观察"总记录数"是否变成 12——这验证了 `a` 与 `w` 的行为差异；再把 `line.strip()` 注释掉，看超速判断是否仍然正确（`int` 能容忍 `\n`，但字符串比较不能）。

## 案例 8-2　交通流量日报表（CSV）生成与汇总

- **覆盖知识点**：8.5（CSV 写入与解析）、8.1（`w` / `a` / `r`）、8.2（`readlines`）、8.3（追加汇总行）
- **源文件**：`case2_csv日报表.py`
- **难度**：综合　**建议用时**：35 分钟

### 任务情境

每日需向管理平台导出 CSV 报表，字段为"时段,车流量,平均车速"，随后读回计算全天总量与加权平均车速。

### 任务要求

1. 用 `",".join()` 把每行数据拼成 CSV 文本写入文件，第一行为表头；
2. 读回时跳过表头，按 `split(",")` 解析；
3. 统计全天总流量、平均车速（按流量加权）；
4. 输出格式化后的汇总行，并追加写入报表末尾。

### 参考代码

In [ ]:
# ---------- 第 08 章 文件操作 · 案例 8-2：交通流量日报表（CSV）生成与汇总 ----------
# 对应知识点：8.5 CSV 手工写入与解析、8.1 w / a / r 三种模式、8.2 readlines()
# 功能说明：生成日报 CSV，读回按流量加权求平均车速，追加合计行后验证文件行数

# ---------- 1) 生成 CSV 报表 ----------
rows = [
    ("时段", "车流量", "平均车速"),
    ("06:00-08:00", 820, 68),
    ("08:00-10:00", 1100, 42),
    ("10:00-12:00", 760, 58),
    ("12:00-14:00", 690, 62),
    ("14:00-16:00", 720, 60),
    ("16:00-18:00", 980, 48),
    ("18:00-20:00", 1650, 32),
]

with open("flow_daily.csv", "w", encoding="utf-8") as f:
    for r in rows:
        f.write(",".join(map(str, r)) + "\n")
print("报表已生成，共", len(rows), "行")

# ---------- 2) 读回并汇总 ----------
total_flow = 0
weighted = 0
first = True

with open("flow_daily.csv", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line == "":
            continue
        if first:                 # 跳过表头
            first = False
            continue
        parts = line.split(",")
        flow = int(parts[1])
        speed = int(parts[2])
        total_flow += flow
        weighted += flow * speed

avg_speed = round(weighted / total_flow, 1)
print("全天总流量：", total_flow, "辆")
print("加权平均车速：", avg_speed, "km/h")

# ---------- 3) 追加汇总行 ----------
with open("flow_daily.csv", "a", encoding="utf-8") as f:
    f.write("合计," + str(total_flow) + "," + str(avg_speed) + "\n")
print("汇总行已追加")

# ---------- 4) 验证追加结果 ----------
with open("flow_daily.csv", "r", encoding="utf-8") as f:
    lines = f.readlines()
print("当前文件行数：", len(lines))
print("最后一行：", lines[-1].strip())

### 运行结果

```text
报表已生成，共 8 行
全天总流量： 6720 辆
加权平均车速： 49.4 km/h
汇总行已追加
当前文件行数： 9
最后一行： 合计,6720,49.4
```

### 代码解读

| 代码段 | 对应知识点 | 作用 |
|---|---|---|
| `rows` 列表含元组 | 5.1、5.3 | 用元组表示一行，结构固定不可改 |
| `",".join(map(str, r))` | 8.5、7.2 | 数字先转字符串才能 `join` |
| `"w"` 模式 | 8.1 | 每次运行重新生成整表 |
| `first = True` 跳过表头 | 8.2 | 标志变量法，适合"只跳第一行" |
| `weighted += flow * speed` | 8.5 | 加权求和：流量 × 车速，避免简单平均失真 |
| `weighted / total_flow` | 8.5 | 加权平均 = 加权和 ÷ 总权重 |
| `"a"` 追加汇总行 | 8.1、8.3 | 若用 `w` 会清空整表 |
| `f.readlines()` + `lines[-1]` | 8.2 | 读回验证，`-1` 取最后一行 |

**改一改**：把第 3 步的 `"a"` 改成 `"w"`，观察"当前文件行数"变成 1、原报表丢失；再把加权平均改成简单平均 `sum(speed) / len(speed)`，比较 49.4 与 52.9 的差异，理解"加权"的业务含义（高峰时段流量大但车速低，简单平均会高估整体通行状况）。

---

# 第三部分　布置练习

## 练习 8-1　分时段过车记录统计

- **对应知识点**：8.1（`w` / `r`）、8.2（逐行解析）、5.5（字典累加）
- **提交文件**：`practice1_分时统计.py`
- **难度**：综合　**建议用时**：30 分钟

### 题目

某检测器一个班次的记录保存在 `flow_log.txt` 中，每行格式为 `小时,车流量`。请读入该文件，按小时汇总车流量并输出总量、峰值小时与平均值。

### 要求

1. 程序先用 `w` 模式生成 9 行样例数据（6~20 时偶数小时，其中 6 时重复两条）；
2. 用 `with` 读取，逐行 `split(",")` 解析；
3. 用字典按小时累加（同一小时可能有多条记录）；
4. 输出总量、峰值小时、平均值（保留 1 位小数）。

### 提示

- 字典累加的关键是 `d[k] = d.get(k, 0) + v`——`get` 的默认值 0 保证了首次出现时不会 `KeyError`。
- 峰值小时可用 `max(hourly, key=lambda h: hourly[h])`：`max` 遍历的是键，`key` 决定比较依据（第 7 章知识点 7.3）。
- 平均值分母是**小时数**（去重后），不是记录条数。

### 自检要点（不含答案）

- [ ] 生成的文件有 9 行，读回后统计的小时数为 8；
- [ ] 6 时的车流量为两条之和，而非其中一条；
- [ ] 峰值小时为 18 时；
- [ ] 平均值以 8 为分母。

## 练习 8-2　告警记录追加写入

- **对应知识点**：8.1（`a` 模式）、8.3（写入换行）、8.2（读回统计）
- **提交文件**：`practice2_告警追加.py`
- **难度**：基础　**建议用时**：25 分钟

### 题目

系统在检测到超速时需把告警追加写入 `alarm.txt`，格式为 `时间,车牌,车速`。请实现：写入 3 条告警后，再追加 2 条，并读回统计告警总条数与最高车速。

### 要求

1. 第一次写入用 `a` 模式（模拟系统长期运行）；
2. 分割成两次写入，验证内容不会被覆盖；
3. 读回时统计条数与最高车速；
4. 输出每条告警的行号与内容。

### 提示

- 若第一次误用 `w`、第二次也用 `w`，文件里只会剩最后 2 条——这是"日志丢失"最常见的原因。用 `a` 两次写入后应有 5 行。
- 连续运行两次程序会得到 10 行，这是 `a` 的正常行为，不是 bug；批改时先删除旧文件。
- 行号从 1 开始计数，用计数器变量在循环内 `+= 1`。

### 自检要点（不含答案）

- [ ] 两次写入后文件为 5 行；
- [ ] 第二次写入未覆盖第一批；
- [ ] 最高车速 150 km/h；
- [ ] 每条输出形如"第1条：…"，行号连续。

---

# 第四部分　习题答案

## 答案 8-1　分时段过车记录统计

> 对应 **练习 8-1**（第三部分）

### 参考代码

In [ ]:
# ---------- 第 08 章 文件操作 · 练习 8-1 参考答案：分时段过车记录统计 ----------
# 对应知识点：8.1 w / r 模式、8.2 逐行解析、5.5 字典按键累加
# 功能说明：把过车记录按小时累加写入并读回，统计分时车流量、峰值小时与小时均值

# 1) 生成样例数据（同一小时故意重复两条，用于验证累加）
sample = [
    (6, 420), (6, 400), (8, 1100), (10, 760),
    (12, 690), (14, 720), (16, 980), (18, 1650), (20, 1320),
]
with open("flow_log.txt", "w", encoding="utf-8") as f:
    for hour, flow in sample:
        f.write(str(hour) + "," + str(flow) + "\n")

# 2) 读取并按小时累加
hourly = {}
with open("flow_log.txt", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line == "":
            continue
        parts = line.split(",")
        hour = int(parts[0])
        flow = int(parts[1])
        hourly[hour] = hourly.get(hour, 0) + flow

# 3) 统计
total = sum(hourly.values())
peak_hour = max(hourly, key=lambda h: hourly[h])
print("参与统计的小时数：", len(hourly))
print("全天总流量：", total, "辆")
print("峰值小时：", peak_hour, "时，流量", hourly[peak_hour], "辆")
print("小时均值：", round(total / len(hourly), 1), "辆")

### 运行结果

```text
参与统计的小时数： 8
全天总流量： 8040 辆
峰值小时： 18 时，流量 1650 辆
小时均值： 1005.0 辆
```

### 解析

1. **9 行数据、8 个小时**：6 时出现两次，被字典合并，因此 `len(hourly)` 是 8 而非 9。这正说明字典是"按小时汇总"的正确容器。
2. **`get(k, 0) + v` 模式**：首次遇到 6 时，`get` 返回 0，`0 + 420 = 420` 写入；第二次 `420 + 400 = 820`。若不写默认值，第一次就抛 `KeyError`。
3. **`max(hourly, key=...)` 的含义**：`max` 遍历字典得到的是**键**（小时），`key` 函数把每个键映射为流量用于比较，最终返回流量最大的**键**。这是"求极值并保留标签"的标准写法。
4. **平均值分母**：`len(hourly)` = 8，与 `len(sample)` = 9 不同。若用 9 作分母，结果变成 893.3，口径错误。

### 常见错误

| 错误现象 | 原因 | 修正 |
|---|---|---|
| `KeyError: 6` | 未用 `get(k, 0)` | 加默认值 |
| 峰值返回 1650 | `max(hourly.values())` | 要小时用 `max(hourly, key=...)` |
| 6 时只算 400 | 直接赋值 `hourly[hour] = flow` | 必须累加 |
| 均值 893.3 | 分母用了记录条数 | 分母为 `len(hourly)` |

### 评分要点（供批改）

| 项 | 分值 | 观察点 |
|---|---|---|
| 文件生成与读取 | 3 | `w` 写、`with` 读，编码正确 |
| 字典累加 | 4 | 同小时合并，结果为 820 |
| 峰值与均值 | 3 | `max(key=)` 用法、分母正确 |

## 答案 8-2　告警记录追加写入

> 对应 **练习 8-2**（第三部分）

### 参考代码

In [ ]:
# ---------- 第 08 章 文件操作 · 练习 8-2 参考答案：告警记录追加写入 ----------
# 对应知识点：8.1 a 追加模式（不覆盖）、8.3 写入与换行、8.2 读回统计
# 功能说明：分两批把告警记录追加写入同一文件，读回校验旧数据未被覆盖并统计最高车速

# 第一批告警
batch1 = [
    "2026-10-29 08:16:32,津C24680,132",
    "2026-10-29 08:17:32,京E55667,150",
    "2026-10-29 09:05:10,冀D11223,128",
]
with open("alarm.txt", "a", encoding="utf-8") as f:
    for item in batch1:
        f.write(item + "\n")

# 第二批告警（追加，不覆盖）
batch2 = [
    "2026-10-29 18:22:05,京A12345,135",
    "2026-10-29 18:30:44,京B67890,142",
]
with open("alarm.txt", "a", encoding="utf-8") as f:
    for item in batch2:
        f.write(item + "\n")

# 读回统计
count = 0
max_speed = 0
with open("alarm.txt", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line == "":
            continue
        count += 1
        parts = line.split(",")
        speed = int(parts[2])
        if speed > max_speed:
            max_speed = speed
        print("第" + str(count) + "条：", line)

print("告警总数：", count, "条")
print("最高车速：", max_speed, "km/h")

### 运行结果

```text
第1条： 2026-10-29 08:16:32,津C24680,132
第2条： 2026-10-29 08:17:32,京E55667,150
第3条： 2026-10-29 09:05:10,冀D11223,128
第4条： 2026-10-29 18:22:05,京A12345,135
第5条： 2026-10-29 18:30:44,京B67890,142
告警总数： 5 条
最高车速： 150 km/h
```

### 解析

1. **`a` 的核心价值**：两次 `open` 各自独立，但都追加到文件末尾，形成 5 行完整历史。日志系统、审计记录都必须用 `a`。
2. **行号即计数器**：`count += 1` 放在解析前，使行号与实际文件行一致（从 1 开始）；若放在 `continue` 之后、解析之前，空行也会占号，造成编号跳空。
3. **极值维护**：`max_speed = 0` 在循环外，循环内比较更新——第 3 章知识点 3.5 的模式在文件处理中同样适用。
4. **重复运行的后果**：`a` 会不断追加，第二次运行后变成 10 行。这是设计意图（日志只增不改），若需幂等应改用 `w` 或在写入前清空文件。

### 常见错误

| 错误现象 | 原因 | 修正 |
|---|---|---|
| 只剩 2 条 | 用了 `w` | 改 `a` |
| 记录挤在一行 | 忘记 `+ "\n"` | 每条加换行 |
| 最高车速 0 | `max_speed` 初值写在循环内 | 移到循环外 |
| 行号从 0 开始 | `count` 自增位置写反 | 先自增再打印 |

### 评分要点（供批改）

| 项 | 分值 | 观察点 |
|---|---|---|
| `a` 模式 | 3 | 两次写入均为追加 |
| 换行与编码 | 2 | 5 行独立、中文正常 |
| 读回统计 | 3 | 条数 5、最高 150 |
| 行号输出 | 2 | 从 1 开始、连续 |

---

## 本章小结

- **模式**：`r` 只读、`w` 清空重写、`a` 追加；**日志一律用 `a`**。
- **读取**：大文件用 `for line in f`；需要按下标访问用 `readlines()`；`read()` 只适合小文件。
- **写入**：`write()` 不自动换行，自己加 `"\n"`；写入内容必须是字符串。
- **规范**：一律 `with open(...) as f` + `encoding="utf-8"`，读写两侧都要写。
- **CSV**：`",".join()` 写、`split(",")` 读，表头要跳过，汇总行用 `a` 追加。

## 自检清单

- [ ] 能说出 `w` 与 `a` 的区别，并解释"日志丢失"是怎么发生的；
- [ ] 会写 `with open(...) as f:` 并在块内完成全部处理；
- [ ] 知道逐行读取的每行带 `\n`，必须先 `strip()`；
- [ ] 会把数字转成字符串再 `join`；
- [ ] 会跳过表头、累加到字典、追加汇总行；
- [ ] 能解释为什么加权平均不等于简单平均。

## 前后章节衔接

- 本章的 `int(parts[2])` 与 `open` 都可能抛异常，**第 9 章**将用 `try-except` 让程序"遇到坏数据不崩溃"。
- 本章的"多文件"（`traffic_utils.py` + 主程序）在**第 10 章**升级为模块与程序包。
- 本章生成的 `flow_daily.csv` 与 `alarm.txt`，在**第 9 章综合考核**中会以 `traffic_data.csv` 的形式再次出现。

---

[🏠 课程首页](../00_课程索引.ipynb)&nbsp;&nbsp;|&nbsp;&nbsp;[◀ 上一章：高级函数](../07高级函数/07_高级函数.ipynb)&nbsp;&nbsp;|&nbsp;&nbsp;[下一章：异常处理 ▶](../09异常处理/09_异常处理.ipynb)